# 🤖 ML-Based Phishing Email Classifier

This notebook implements a machine learning pipeline for phishing detection:
1. **Consumes** emails from Kafka stream
2. **Classifies** using trained ML model (TF-IDF + Logistic Regression)
3. **Maps** detections to MITRE ATT&CK framework
4. **Saves** results for analysis

**MITRE ATT&CK Techniques Covered**:
- T1566.001 - Spearphishing Attachment
- T1566.002 - Spearphishing Link
- T1598 - Phishing for Information

In [1]:
!pip install kafka-python pandas scikit-learn jaeger-client joblib

In [2]:
import json
import pandas as pd
import numpy as np
from kafka import KafkaConsumer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from jaeger_client import Config
from datetime import datetime
import joblib
import os
import re

In [3]:
# Initialize Jaeger tracer
config = Config(
    config={
        'sampler': {'type': 'const', 'param': 1},
        'logging': True,
    },
    service_name='ml-classifier',
)
tracer = config.initialize_tracer()
print("✅ Jaeger tracer initialized!")

✅ Jaeger tracer initialized!


## 🎯 MITRE ATT&CK Mapping

We map phishing detections to specific MITRE ATT&CK techniques based on email characteristics.

In [4]:
# MITRE ATT&CK Mapping for Phishing
MITRE_MAPPING = {
    'phishing_link': {
        'tactic': 'Initial Access',
        'technique_id': 'T1566.002',
        'technique_name': 'Phishing: Spearphishing Link',
        'description': 'Adversaries send spearphishing emails with malicious links',
        'severity': 'high'
    },
    'phishing_credential': {
        'tactic': 'Credential Access',
        'technique_id': 'T1598.003',
        'technique_name': 'Phishing for Information: Spearphishing Link',
        'description': 'Adversaries send phishing messages to elicit sensitive information',
        'severity': 'critical'
    },
    'phishing_attachment': {
        'tactic': 'Initial Access',
        'technique_id': 'T1566.001',
        'technique_name': 'Phishing: Spearphishing Attachment',
        'description': 'Adversaries send spearphishing emails with malicious attachments',
        'severity': 'high'
    },
    'social_engineering': {
        'tactic': 'Initial Access',
        'technique_id': 'T1566',
        'technique_name': 'Phishing',
        'description': 'Generic phishing attempt using social engineering',
        'severity': 'medium'
    },
    'legitimate': {
        'tactic': None,
        'technique_id': None,
        'technique_name': 'Legitimate Email',
        'description': 'No threat detected',
        'severity': 'none'
    }
}

# Indicators for sub-classification
CREDENTIAL_KEYWORDS = ['password', 'verify', 'account', 'login', 'credential', 'ssn', 'social security', 'bank detail']
URGENCY_KEYWORDS = ['urgent', 'immediate', 'now', 'today', '24 hours', 'suspended', 'terminated', 'final warning']
LINK_PATTERNS = [r'http[s]?://[^\s]+\.tk', r'http[s]?://[^\s]+\.xyz', r'http[s]?://[^\s]+\.ru', r'http[s]?://[^\s]+\.biz']

In [5]:
def extract_features(text):
    """
    Extract additional features from email text for analysis.
    """
    text_lower = text.lower()
    
    features = {
        'has_suspicious_link': any(re.search(pattern, text_lower) for pattern in LINK_PATTERNS),
        'has_urgency': any(keyword in text_lower for keyword in URGENCY_KEYWORDS),
        'has_credential_request': any(keyword in text_lower for keyword in CREDENTIAL_KEYWORDS),
        'link_count': len(re.findall(r'http[s]?://', text_lower)),
        'exclamation_count': text.count('!'),
        'uppercase_ratio': sum(1 for c in text if c.isupper()) / max(len(text), 1)
    }
    
    return features

def get_mitre_classification(email_text, is_phishing):
    """
    Map detected phishing to specific MITRE ATT&CK technique.
    """
    if not is_phishing:
        return MITRE_MAPPING['legitimate']
    
    features = extract_features(email_text)
    
    # Determine specific phishing type
    if features['has_credential_request']:
        return MITRE_MAPPING['phishing_credential']
    elif features['has_suspicious_link']:
        return MITRE_MAPPING['phishing_link']
    else:
        return MITRE_MAPPING['social_engineering']

print("✅ Feature extraction functions defined!")

✅ Feature extraction functions defined!


## 🧠 Machine Learning Model Training

We use **TF-IDF** (Term Frequency-Inverse Document Frequency) for text vectorization and **Logistic Regression** for classification.

In [6]:
# Load training data
df = pd.read_csv('/home/jovyan/data/emails.csv')

# Combine subject and body for analysis
df['full_text'] = df['subject'] + ' ' + df['body']

# Convert labels to binary
df['is_phishing'] = (df['label'] == 'phishing').astype(int)

print(f"📊 Dataset loaded: {len(df)} emails")
print(f"   - Phishing: {df['is_phishing'].sum()}")
print(f"   - Legitimate: {len(df) - df['is_phishing'].sum()}")

📊 Dataset loaded: 100 emails
   - Phishing: 37
   - Legitimate: 63


In [7]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['full_text'], 
    df['is_phishing'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['is_phishing']
)

print(f"Training set: {len(X_train)} emails")
print(f"Test set: {len(X_test)} emails")

Training set: 80 emails
Test set: 20 emails


In [8]:
# Create TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2),  # Use both unigrams and bigrams
    min_df=2
)

# Fit and transform training data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"✅ TF-IDF Vectorizer created!")
print(f"   Vocabulary size: {len(vectorizer.vocabulary_)}")

✅ TF-IDF Vectorizer created!
   Vocabulary size: 172


In [9]:
# Train Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)

model.fit(X_train_tfidf, y_train)
print("✅ Model trained!")

✅ Model trained!


In [10]:
# Evaluate model
y_pred = model.predict(X_test_tfidf)

print("\n" + "="*50)
print("📊 MODEL EVALUATION")
print("="*50)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


📊 MODEL EVALUATION

Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00        13
    Phishing       1.00      1.00      1.00         7

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[13  0]
 [ 0  7]]


In [11]:
# Save model and vectorizer
joblib.dump(model, '/home/jovyan/models/phishing_model.pkl')
joblib.dump(vectorizer, '/home/jovyan/models/tfidf_vectorizer.pkl')
print("✅ Model and vectorizer saved to /models/")

✅ Model and vectorizer saved to /models/


In [12]:
# Show top features for phishing detection
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

# Top phishing indicators
top_phishing_idx = np.argsort(coefficients)[-15:]
print("\n🚨 Top Phishing Indicators:")
for idx in reversed(top_phishing_idx):
    print(f"   {feature_names[idx]}: {coefficients[idx]:.3f}")

# Top legitimate indicators
top_legit_idx = np.argsort(coefficients)[:10]
print("\n✅ Top Legitimate Indicators:")
for idx in top_legit_idx:
    print(f"   {feature_names[idx]}: {coefficients[idx]:.3f}")


🚨 Top Phishing Indicators:
   http: 1.389
   account: 1.016
   claim: 1.007
   verify: 0.950
   transfer: 0.716
   ve: 0.625
   immediately: 0.614
   000: 0.562
   send: 0.490
   com: 0.488
   download: 0.484
   irs: 0.478
   password: 0.476
   million: 0.457
   details: 0.446

✅ Top Legitimate Indicators:
   new: -0.802
   reminder: -0.585
   request: -0.500
   ready: -0.479
   attached: -0.477
   order: -0.410
   thank: -0.403
   review: -0.392
   days: -0.375
   book: -0.357


## 📥 Kafka Consumer & Real-Time Classification

In [13]:
def classify_email(email_event, model, vectorizer, tracer):
    """
    Classify an email using the trained ML model and map to MITRE ATT&CK.
    """
    with tracer.start_span('classify_email') as span:
        # Combine subject and body
        full_text = f"{email_event['subject']} {email_event['body']}"
        
        # Vectorize
        text_tfidf = vectorizer.transform([full_text])
        
        # Predict
        prediction = model.predict(text_tfidf)[0]
        probability = model.predict_proba(text_tfidf)[0]
        
        is_phishing = bool(prediction)
        confidence = probability[1] if is_phishing else probability[0]
        
        # Get MITRE mapping
        mitre_info = get_mitre_classification(full_text, is_phishing)
        
        # Extract additional features
        features = extract_features(full_text)
        
        span.set_tag('is_phishing', is_phishing)
        span.set_tag('confidence', f"{confidence:.2%}")
        span.set_tag('mitre_technique', mitre_info['technique_id'])
        
        # Build classified event
        classified_event = {
            **email_event,
            'classification': 'phishing' if is_phishing else 'legitimate',
            'confidence': float(confidence),
            'mitre_tactic': mitre_info['tactic'],
            'mitre_technique_id': mitre_info['technique_id'],
            'mitre_technique_name': mitre_info['technique_name'],
            'severity': mitre_info['severity'],
            'has_suspicious_link': features['has_suspicious_link'],
            'has_urgency': features['has_urgency'],
            'has_credential_request': features['has_credential_request'],
            'classified_at': datetime.utcnow().isoformat()
        }
        
        return classified_event

In [14]:
def consume_and_classify(max_messages=100, timeout_ms=10000):
    """
    Consume emails from Kafka and classify them using ML model.
    """
    # Load model
    model = joblib.load('/home/jovyan/models/phishing_model.pkl')
    vectorizer = joblib.load('/home/jovyan/models/tfidf_vectorizer.pkl')
    print("✅ Model loaded!")
    
    # Initialize Kafka consumer
    consumer = KafkaConsumer(
        'emails.raw',
        bootstrap_servers=['kafka:9092'],
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='earliest',
        enable_auto_commit=True,
        group_id='phishing-classifier-group',
        consumer_timeout_ms=timeout_ms
    )
    print("✅ Kafka consumer connected!")
    print(f"\n📥 Consuming up to {max_messages} messages...\n")
    
    classified_emails = []
    phishing_count = 0
    legitimate_count = 0
    
    try:
        for message in consumer:
            email_event = message.value
            
            # Classify
            classified = classify_email(email_event, model, vectorizer, tracer)
            classified_emails.append(classified)
            
            # Count
            if classified['classification'] == 'phishing':
                phishing_count += 1
                print(f"🚨 PHISHING DETECTED: {classified['subject'][:50]}... "
                      f"[{classified['mitre_technique_id']}] ({classified['confidence']:.1%})")
            else:
                legitimate_count += 1
            
            if len(classified_emails) >= max_messages:
                break
                
    except Exception as e:
        print(f"Consumer stopped: {e}")
    finally:
        consumer.close()
    
    # Save results
    if classified_emails:
        df_results = pd.DataFrame(classified_emails)
        df_results.to_csv('/home/jovyan/data/classified_emails.csv', index=False)
        print(f"\n✅ Saved {len(classified_emails)} classified emails to CSV")
    
    print(f"\n" + "="*50)
    print("📊 CLASSIFICATION SUMMARY")
    print("="*50)
    print(f"Total processed: {len(classified_emails)}")
    print(f"Phishing detected: {phishing_count} ({phishing_count/max(len(classified_emails),1):.1%})")
    print(f"Legitimate: {legitimate_count} ({legitimate_count/max(len(classified_emails),1):.1%})")
    
    return classified_emails

In [15]:
# Run the classifier
classified_emails = consume_and_classify(max_messages=100, timeout_ms=15000)

✅ Model loaded!
✅ Kafka consumer connected!

📥 Consuming up to 100 messages...

🚨 PHISHING DETECTED: URGENT: Password expires today... [T1598.003] (61.9%)
🚨 PHISHING DETECTED: Claim your free iPhone 15!... [T1566] (58.5%)
🚨 PHISHING DETECTED: ALERT: Unusual sign-in activity... [T1598.003] (66.1%)
🚨 PHISHING DETECTED: Bitcoin investment opportunity... [T1566] (59.0%)
🚨 PHISHING DETECTED: You've received a fax!... [T1566] (68.6%)
🚨 PHISHING DETECTED: Urgent: Confirm wire transfer... [T1598.003] (69.5%)
🚨 PHISHING DETECTED: You've been selected for a survey... [T1566] (61.5%)
🚨 PHISHING DETECTED: Your loan is pre-approved!... [T1566] (64.7%)
🚨 PHISHING DETECTED: Immediate action required - IRS... [T1566] (70.1%)
🚨 PHISHING DETECTED: URGENT: Verify PayPal account... [T1598.003] (77.8%)
🚨 PHISHING DETECTED: Urgent: Your account has been compromised... [T1598.003] (81.1%)
🚨 PHISHING DETECTED: IRS Tax Refund Notification... [T1566] (73.9%)
🚨 PHISHING DETECTED: Verify your Apple ID... [T1598.0

In [16]:
# Check accuracy against original labels
if classified_emails:
    df_results = pd.DataFrame(classified_emails)
    
    # Compare predictions with original labels
    df_results['correct'] = (
        (df_results['classification'] == 'phishing') == 
        (df_results['original_label'] == 'phishing')
    )
    
    accuracy = df_results['correct'].mean()
    
    print(f"\n🎯 Pipeline Accuracy: {accuracy:.2%}")
    print(f"\n📊 MITRE ATT&CK Techniques Detected:")
    mitre_counts = df_results[df_results['mitre_technique_id'].notna()]['mitre_technique_id'].value_counts()
    for technique, count in mitre_counts.items():
        print(f"   {technique}: {count} detections")


🎯 Pipeline Accuracy: 100.00%

📊 MITRE ATT&CK Techniques Detected:
   T1598.003: 18 detections
   T1566: 16 detections
   T1566.002: 3 detections
